# **Simulation Tool 2026**

## Install dependencies

In [ ]:
# %pip install rocketpy==1.12.1
# %pip install CoolProp
# %pip install openmeteo-requests requests-cache retry-requests pandas plotly colorama
# %pip install kaleido
# %pip install --upgrade nbformat
# %pip install "niquests==3.18.8" "urllib3-future==2.20.904"
# %pip install nbstripout
# %pip install nbconvert
# %pip install pyproj pymap3d

## Imports

In [ ]:
import simulation.simulation as sim
import simulation.outputs as outputs
import simulation.utils as utils
import simulation.reanalysis as reanalysis
from pathlib import Path

In [ ]:
# reload imported Python modules when you change their .py files
%load_ext autoreload
%autoreload 2

## Configuration
The length unit chosen here is millimeters to keep the values more readable. If necessary, values should be converted accordingly.

`OUTPUT_LEVEL = 0`      (standard): most important prints + landing/safety plots \
`OUTPUT_LEVEL = 1`      (detailed): level 0 + per-flight custom plots + env prints \
`OUTPUT_LEVEL = 2`    (more detailed): level 1 + rocket/motor prints \
`OUTPUT_LEVEL = 3`    (debug): level 2 + more env plots

In [ ]:
# TODO: config generation

# select project
PROJECT = "ALBATROSS"
CONFIG_PATH = Path(f"{PROJECT}/config.json")
ZONES_PATH = Path(f"{PROJECT}/zones.json")

# TODO: rather put switches into the config to later be able to select it in the GUI
OUTPUT_LEVEL = 2

# True  -> use ONLY the standard atmosphere
# False -> use every envType from config EXCEPT the standard atmosphere (Windy / custom / etc.).
USE_ONLY_STANDARD_ENVIRONMENT = False

constants, variations, export_kml = utils.load_config(CONFIG_PATH)
constants["project_path"] = Path(PROJECT)

# Override the envType list from config based on the switch above.
if USE_ONLY_STANDARD_ENVIRONMENT:
    constants["environment_envType"] = ("standard_atmosphere",)
else:
    constants["environment_envType"] = tuple(
        env_type for env_type in constants["environment_envType"] if env_type != "standard_atmosphere"
    )

if variations:
    print("Variations:")
    # values with a range start..stop:step
    for name in variations:
        print(f"- {name}")

print("\nConstants loaded:", len(constants))
# for name in constants:
#     print(f"- {name}")
print("\nActive envTypes:", constants["environment_envType"])

## Zones

In [ ]:
# TODO: move this into utils and only surface really necessary code here
# Define polygons (x = WO, y = NS)
exclusion_zones, buffer_zones, exclusion_zone_saftey_margin = utils.load_zones(ZONES_PATH)

# Add buffer zones around exclusion zones
buffer_zones.update(utils.scale_zones(exclusion_zones, exclusion_zone_saftey_margin))

constants, variations = utils.register("exclusion_zones", exclusion_zones, constants, variations)
constants, variations = utils.register("buffer_zones", buffer_zones, constants, variations)

# buffer_zones = {}                     # remove all buffer zones; check only hard exclusions
# buffer_zones = exclusion_zones        # use exclusions as buffers (no margin, discard buffer zones from zones.json)

outputs.plot_landing_positions_with_modes(constants, exclusion_zones, buffer_zones, plot_name="Zones", zones_only=True, save_format="png")

## Environments Initialization
In this section the environments are initialized.

In [ ]:
constants, variations = sim.create_environment(constants, variations, OUTPUT_LEVEL)

## Simulation
### Tanks / Engine

In [ ]:
constants, variations = sim.create_engine(constants, variations, OUTPUT_LEVEL)

### Rocket
RocketPy definitions:
- dry mass = rocket with motor but without propellant
- Rocket Loaded Mass = Wet mass
- Rocket Center of Dry Mass - Nozzle Exit = Rocket Center of Dry Mass from bottom

In [ ]:
constants, variations = sim.create_rocket(constants, variations, OUTPUT_LEVEL)

### Flight

In [ ]:
constants, variations = sim.create_flight(constants, variations)

## Output

A heading direction is only safe if <u>every</u> simulated rocket scenario (nominal/no main/ballistic/payload), every inclination and every environment stays outside the buffer zones.

In [ ]:
reanalysis.build_reanalysis_artifacts(constants, variations)

constants, variations = outputs.run_notebook_display_mode(
    constants,
    variations,
    exclusion_zones,
    buffer_zones,
    OUTPUT_LEVEL=OUTPUT_LEVEL,
)

reanalysis.run_reanalysis_comparison(constants, variations)

if export_kml:
    exported_kml_files = outputs.export_all_kml(constants, variations)
    # exported_csv_files = outputs.export_all_trajectory_csv(constants, variations)         # by request for WARR, maybe useful, otherwise remove later

outputs.export_notebook_to_html("simulation_orchestration.ipynb", constants["project_path"] / "report.html")